In [ ]:
import lsdb
from dask.distributed import Client
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import multiband_fit_template_numba as mbft
import nested_pandas as npd
from nested_pandas.utils import count_nested
from scipy.signal import find_peaks

import logging

## Setup and DP2 Loading

In [ ]:
client=Client(n_workers=6, memory_limit="24GB", threads_per_worker=1, silence_logs=logging.ERROR)
client

In [ ]:
cone = lsdb.ConeSearch(ra=300.0, dec=-25.0, radius_arcsec=70000.0)
usdf_path = "/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_6/object_collection"
shire_path = "/astro/store/shire/hats/dash/hats/v30_0_6/object_collection"
dp2 = lsdb.open_catalog(shire_path,
                        columns=['objectId', 'coord_dec', 'coord_ra', 'objectForcedSource', 'u_psfMag', 'u_psfMagErr',
                                 'g_psfMag', 'g_psfMagErr', 'r_psfMag', 'r_psfMagErr', 'i_psfMag', 'i_psfMagErr', 'z_psfMag', 'z_psfMagErr',
                                 'y_psfMag', 'y_psfMagErr', 'ebv'],
                        search_filter=cone)

dp2

# 1. Filter to RR Lyrae Candidate Subset

In [ ]:
# load in the fitter template
tem = mbft.load_template_dir("lsst_template")

# Define Flag Columns
nested_cols = dp2.meta.get_subcolumns("objectForcedSource")
FLAG_COLS = [col for col in nested_cols if 'Flag' in col or 'flag' in col]

In [ ]:
# a bunch of helper functions to help us whittle down the big data set
# this one does what it says on the tin
def count_points(df, new_name='n_lc'):
    # Asked to count `lc`, this will add a column called `n_lc`
    return count_nested(df, "objectForcedSource").rename(columns={'n_objectForcedSource':new_name})

def calc_std_partition(df, band='g'):
    df = npd.NestedFrame(df)
    
    # filter to requested band (flags already removed by filter_flags_partition)
    lc = df.query(f"objectForcedSource.band == '{band}'")['objectForcedSource']
    
    if len(lc) == 0:
        return df.assign(std=np.nan)
    
    # vectorized min/max over all objects at once
    std = lc['psfMag'].groupby(level=0).std()
    
    # objects with no observations in this band get nan automatically
    std.name = 'std'
    return df.join(std)

def dust_correction_single_band(df, band):
    A_band = df['ebv'] * tem['dust'][band]
    corrected = (df[f'{band}_psfMag'] - A_band)
    df[f'{band}_psfMagExt'] = corrected
    return df

def dust_correction(df):    
    for band in tem['dust'].keys():
        df = dust_correction_single_band(df, band)
    return df

# Compiles various filtering functions into one
def filtering(df):
    # dust correction
    df = dust_correction(df)

    # color cuts
    ug_query = 'u_psfMagExt - g_psfMagExt > 0.500 and u_psfMagExt - g_psfMagExt < 1.4'
    gr_query = 'g_psfMagExt - r_psfMagExt > -0.15 and g_psfMagExt - r_psfMagExt < 0.4'
    ri_query = 'r_psfMagExt - i_psfMagExt > -0.25 and r_psfMagExt - i_psfMagExt < 0.3'
    iz_query = 'i_psfMagExt - z_psfMagExt > -0.21 and i_psfMagExt - z_psfMagExt < 0.45'
    zy_query = 'z_psfMagExt - y_psfMagExt > -0.22 and z_psfMagExt - y_psfMagExt < 0.15'
    color_query = f'{ug_query} and {gr_query} and {ri_query} and {iz_query} and {zy_query}'
    df = df.query(color_query)

    # quality flag cuts
    flag_query = " and ".join(f"{col} == False" for col in FLAG_COLS)
    df = df.query(flag_query)

    # lc length cut
    df = count_points(df).query("n_lc >= 20")

    # variability cut
    df = calc_std_partition(df).query('std >= 0.1')
    return df

In [ ]:
filtered = dp2.map_partitions(filtering)
filtered = filtered.prune_empty_partitions()

# prune columns for downstream memory management
#filtered = filtered.drop(FLAG_COLS)
filtered

In [ ]:
%%time
# check number of surviving objects
def check_len(df):
    return len(df)
lens = filtered.map_partitions(check_len).compute()
lens["result"].sum()

## 2. Run Fitting

In [ ]:
def template_fitting(tem, lc, print_outputs = False, fit_n = 20, coeff_n = 10, omega_n = 20, period_range=[0.2, 0.9], cols=['midpointMjdTai', 'band', 'psfMag', 'psfMagErr']):
    '''
    Returns a dictionary with coeffs (mu, d, a, phi), pests (top 3), cov (of linear params: mu (distance modulus), d (dust), a (amplitude)), sigma_P (local uncertainty of a given period), 
        like_P (global posterior likelihood uncertainty), the next best 3 periods, gen_lc (the generated light curve)
    '''
    # compute best coefficients
    omegas = np.arange(1/period_range[-1], 1/period_range[0], 0.1/omega_n) #periods from [0.2, 0.9]: frequencies from [1.1, 5.0]

    lc_bands = list(lc[cols[1]].unique())
    # print(f"template fitting lc_bands: {lc_bands}")
    
    
    rss = mbft.FitTemplate_multiband(tem, lc, omegas, NN=fit_n, use_errors=True, use_dust=False, use_band_shift=True, cols = cols)
    rss = np.array(rss)
    
    best_omega = omegas[np.argmin(rss)]
    best_pest = 1/best_omega
    
    coeffs, cov = mbft.ComputeCoeffsAndCov_multiband(tem, lc, float(best_omega), NN=coeff_n, use_errors=True, use_dust=False, cols = cols)

    # calculate error and posterior on period
    chi2_min = np.min(rss) # rss is chi2 bc it's already scaled by weights

    chi2_red = chi2_min / (len(lc.midpointMjdTai) - len(coeffs)) # length - dof
        
    periods = 1/np.array(omegas)
    rms = np.sqrt(rss / (len(lc.midpointMjdTai) - len(coeffs))) # length - dof
    mask = rss <= chi2_min + 2.3
   
    sigma_P = 0.5 * np.abs(periods[mask][0] - periods[mask][-1])

    # now get posterior likelihood to figure out how likely this is to be the right answer
    peaks, props = find_peaks(-rss, prominence=10, width=0.1)
    try:
        next_best_idx = np.argpartition(props['prominences'], -3)[-3:]
        next_best = periods[peaks[next_best_idx]]
        next_best_chi = rss[peaks[next_best_idx]]
    except:
        next_best = []

    L = np.exp(-0.5 * (rss - rss.min()))  # subtract min to prevent underflow
    norm = np.trapezoid(L, periods) 
    if norm == 0 or ~np.isfinite(norm):
        like_P = np.nan
    else:
        L /= norm # prob density
        P_mean = np.trapezoid(periods * L, periods) # mean of posterior
        P_var  = np.trapezoid((periods - P_mean)**2 * L, periods) # variance on posterior
        like_P = np.sqrt(P_var) # std/likelihood of posterior

    if print_outputs:
        print("omega_best:", best_omega)
        print(f"pest: {best_pest:0.4f} +- {sigma_P:0.6f}, global uncertainty: {like_P:0.4f}")
        print("coeffs (mu, d, a, phi):", coeffs)  # [mu, d, a, phi]
        # print("cov (mu, d, a):\n", cov)

    return dict({'coeffs':coeffs, 'p_est':best_pest, 'cov':cov, 'variance':sigma_P, 'posterior':like_P, 'next_best':next_best, 'rss':rss})

# fit_n is the number of steps used for fitting the period; coeff_n is the number of steps used when finding coefficients
# omega_n is the number of 0.1 period bins for finding the best period
BANDS = ['u', 'g', 'r', 'i', 'z', 'y']  # fixed order, defined at module level

def fit_stat_df(cdf, nested_column='objectForcedSource', fit_n=20, coeff_n=10, omega_n=20, period_range=(0.2, 0.9)):

    # scalar output columns
    for col in ["wrms", "p_est", "p_err", "wrms_ratio", "mu", "d", "a", "phi"]:
        cdf[col] = np.nan

    for band in BANDS:
        for prefix in ["chi2", "flat_chi2", "wrms", "offset"]:
            cdf[f"{prefix}_{band}"] = np.nan
    cdf["chi2_total"] = np.nan
    cdf["flat_chi2_total"] = np.nan

    # per-band columns — one per band per stat
    # discover bands from the first valid lc
    all_bands = []
    for _, row in cdf.iterrows():
        lc_raw = row[nested_column].dropna(subset=['psfMag', 'psfMagErr', 'midpointMjdTai'])
        bands = sorted(lc_raw['band'].unique())
        if bands:
            all_bands = bands
            break

    for band in all_bands:
        for prefix in ["chi2", "flat_chi2", "wrms", "offset"]:
            cdf[f"{prefix}_{band}"] = np.nan
    cdf["chi2_total"] = np.nan
    cdf["flat_chi2_total"] = np.nan

    for idx, row in cdf.iterrows():
        lc_raw = row[nested_column].dropna(subset=['psfMag', 'psfMagErr', 'midpointMjdTai'])
        flag_cols = [c for c in lc_raw.columns if 'flag' in c.lower()]
        lc_flag = lc_raw[~lc_raw[flag_cols].any(axis=1)]

        band_counts = lc_flag.groupby('band')['psfMag'].count()
        valid_bands = band_counts[band_counts >= 3].index
        lc = lc_flag[lc_flag['band'].isin(valid_bands)].reset_index(drop=True)
        lc_bands = list(lc['band'].sort_values(kind='stable').unique())

        if len(lc_bands) <= 1 or len(lc) < 10:
            continue

        pipeline_output = template_fitting(
            tem, lc, fit_n=fit_n, coeff_n=coeff_n, omega_n=omega_n,
            period_range=list(period_range), print_outputs=False,
        )

        coeffs = pipeline_output['coeffs']
        pest   = pipeline_output['p_est']

        cdf.loc[idx, 'p_est'] = pest
        cdf.loc[idx, 'p_err'] = pipeline_output['posterior']
        cdf.loc[idx, 'mu']    = coeffs[0]
        cdf.loc[idx, 'd']     = coeffs[1]
        cdf.loc[idx, 'a']     = coeffs[2]
        cdf.loc[idx, 'phi']   = coeffs[3]

        tem_plot    = mbft.reorder_template_for_lc(tem, lc_bands)
        gamma       = tem_plot['templates']
        t           = tem_plot['temp_time']
        abs_mag_est = tem_plot['abs_mag'](pest, tem_plot)[0]
        model_err   = tem['model_error']['g']

        chi2_total      = 0.0
        flat_chi2_total = 0.0
        wrms_list       = []
        scatter_ratios  = []
        band_sizes      = []

        for i, band in enumerate(lc_bands):
            offset  = coeffs[4 + i] if i < len(lc_bands) - 1 else 0.0
            m_est   = coeffs[0] + abs_mag_est[i] + coeffs[2] * gamma[i, :] + offset
            band_lc = lc[lc['band'] == band]
            minterp = np.interp(band_lc.midpointMjdTai % pest / pest, t, m_est)

            err_total = np.sqrt(band_lc.psfMagErr ** 2 + model_err ** 2)
            weights   = 1.0 / err_total ** 2

            resid  = band_lc.psfMag - minterp
            chi2   = float(np.sum(resid ** 2 * weights))
            chi2_total += chi2

            flat       = np.average(band_lc.psfMag, weights=weights)
            flat_resid = band_lc.psfMag - flat
            flat_chi2  = float(np.sum(weights * flat_resid ** 2))
            flat_chi2_total += flat_chi2

            wrms_t = float(np.sqrt(np.average(resid ** 2,      weights=weights)))
            wrms_f = float(np.sqrt(np.average(flat_resid ** 2, weights=weights)))
            wrms_list.append(wrms_t)
            scatter_ratios.append(wrms_t / wrms_f)
            band_sizes.append(len(band_lc))

            cdf.loc[idx, f"chi2_{band}"]      = chi2
            cdf.loc[idx, f"flat_chi2_{band}"] = flat_chi2
            cdf.loc[idx, f"wrms_{band}"]      = wrms_t
            cdf.loc[idx, f"offset_{band}"]    = offset

        dof      = len(lc) - len(coeffs)
        flat_dof = len(lc) - len(lc_bands)
        cdf.loc[idx, 'chi2_total']      = chi2_total / dof
        cdf.loc[idx, 'flat_chi2_total'] = flat_chi2_total / flat_dof
        cdf.loc[idx, 'wrms']            = float(np.average(wrms_list,     weights=band_sizes))
        cdf.loc[idx, 'wrms_ratio']      = float(np.average(scatter_ratios, weights=band_sizes))

    cdf = cdf.drop(columns="objectForcedSource")

    return cdf

In [ ]:
# Meta finding: define the output dataframe by running on a small subset
meta_finder = filtered.head(5) # head seems to take longer for larger datasets, for some reason
meta = fit_stat_df(meta_finder)

meta.to_parquet("meta.parquet")
meta

In [ ]:
meta=npd.read_parquet("meta.parquet").drop(columns="_healpix_29")
fitted = filtered.map_partitions(fit_stat_df, meta=npd.NestedFrame(
        {col: pd.Series(dtype=dt) for col, dt in meta.dtypes.items()}
    ))
fitted

# 3. Write out Results

In [ ]:
# Save to catalog
# cone search radius = 7000
#fitted.write_catalog("../dp2_fitted_small", overwrite=True)

# cone search radius = 70000
fitted.write_catalog("../dp2_fitted_medium", overwrite=True)

In [ ]:
# To verify locally -- watch out for results that are too large
result = fitted.compute()
result